# 09 실제 Ollama 요청의 Langfuse 추적

**상황:** 반품·교환 상담 AI가 왜 잘못 답했는지 요청 하나를 추적합니다.

**20분 핵심:** 2–5단계. 1단계는 SDK 사용법, 6–8단계는 비교·검토·기록 확장입니다.
실제 모델 호출: 1단계 1회, 전체 요청 3건 × 생성·평가 2회 = 총 7회.
평가 코드는 VS Code에서 실행되고, Docker Langfuse는 기록을 저장·표시합니다.

**수업 전 준비:** [Docker·VS Code 안내](../langfuse/README.md)의 명령을 터미널에서 마친 뒤 실행합니다.

```sh
python -m pip install -r requirements/week11-langfuse.txt
python week11/lab/langfuse/init_env.py
docker compose -f week11/lab/langfuse/compose.yaml up -d --wait --wait-timeout 300
ollama pull qwen2.5:1.5b-instruct
```

Ollama 앱과 Docker Desktop이 실행 중이어야 합니다. [Langfuse 화면](http://localhost:3011)에 접속하고
`week11/lab/langfuse/.env`의 `LANGFUSE_LOGIN_EMAIL`, `LANGFUSE_LOGIN_PASSWORD`로 로그인합니다.
이 파일은 본인 PC의 비공개 설정입니다. 노트북 출력이나 GitHub에 키·비밀번호를 복사하지 않습니다.

`.venv` Python 3.11 커널에서 위부터 Shift+Enter. 다른 Notebook의 변수는 필요 없습니다.
모델의 의미 오류는 관찰 대상이며, 연결 실패·파싱 오류를 가짜 성공으로 대체하지 않습니다.


**위에서부터 코드 셀을 세는 실행 지도**

VS Code 왼쪽 `[숫자]`는 실행 횟수입니다. 아래 번호는 설명 셀을 제외한 코드 셀 순서입니다. 첫 셀부터 한 개씩 실행하고 완료 후 다음 셀로 이동합니다.

| 코드 셀 | 단계 | 화면에서 누를 곳 | 확인 결과 |
|---|---|---|---|
| 1 | 환경 연결 | Langfuse 로그인 → Week 11 Ollama | 첫 코드 셀의 Python 3.11·모델 확인. 오류가 없으면 다음 셀로 이동. |
| 2 | 1. 최소 SDK 예제 | 출력 ‘실제 생성 요청 열기’ → Tree → ollama-generation → Preview | mini-ollama-request 아래 generation 1개. 실제 Input/Output·모델·토큰 확인. 저장 전이면 몇 초 뒤 같은 링크 새로고침. |
| 3 | 2. 전체 요청 | 출력 ‘Q03a 전체 요청 열기’ | 교환 기간·비용 질문의 실제 답변 생성. 다음 코드 셀까지 실행한 뒤 화면 확인. |
| 4 | 3. 서버 저장 확인 | Tree → 01-retrieval → Output; 02-ollama-answer → Input/Output | ‘실제 부모 1개 + 하위 작업 4개 확인’. observations 5개, generation 2개. 생성 입력에 참고 정답이 없는지 확인. |
| 5 | 4. 점수와 근거 | Tree의 rag-Q03a-grounded 선택 → Scores; 04-ollama-judge → Preview | 코드 검사와 llm_groundedness 분리. 교환 7일·재고 조건·왕복 6000원과 실제 답변 대조. True나 supported는 의미 정확성 보증이 아님. |
| 6 | 5. 재고 질문 비교 | 출력 baseline·grounded 요청 링크를 각각 열기 | 검색 문서에는 실시간 재고가 없음. 답변 문장·citations·abstain·judge 판정을 비교. |
| 7 | 6. 사람 검토 | grounded 요청 → Scores → human_review | 답변을 읽은 뒤 REVIEW와 COMMENT 입력 후 실행. None이면 ‘미실행’이 정상이며 점수가 생기지 않음. |
| 8 | 7. 결과 저장 | VS Code → week11/lab/reports/langfuse/lesson09-traces.json | 요청 3건 저장. 공개 GitHub에 자동 업로드되지 않음. |
| 9 | 8. 개선 정리 | VS Code 결과표 | 관찰·검토·다음 실험의 연결 확인. 이후 10번 파일로 이동. |

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'week11/lab/langfuse_lab.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert sys.version_info[:2] == (3, 11), 'Python 3.11 .venv 커널을 선택하세요.'
import pandas as pd
from IPython.display import display, Markdown
from week11.lab.evaluation import load_data
from week11.lab.ollama_lab import OllamaClient
from week11.lab.langfuse_lab import connect_langfuse, traced_question, wait_for_trace, observation_table, score_table
lf = connect_langfuse()  # 이 실습 폴더의 .env만 사용. 키를 출력하지 않음.
ollama = OllamaClient()
print(ollama.probe())
by_id = {q['id']: q for q in load_data()[1]}


## 1. with 문으로 직접 기록하는 실제 생성

전체 프레임워크 전에 SDK의 최소 구문을 실행합니다. 질문·답변·토큰이 화면에 어떻게 대응되는지 확인합니다.

In [ ]:
from week11.lab.ollama_lab import ANSWER_SCHEMA
with lf.start_as_current_observation(name='mini-ollama-request', input='반품 기간은?') as root:
    with lf.start_as_current_observation(name='ollama-generation', as_type='generation', model=ollama.model) as generation:
        messages = [{'role':'user','content':'정책: 수령 후 7일 이내 반품 접수. 질문: 반품 기간은? JSON answer, citations=[], abstain=false로 답하라.'}]
        actual = ollama.chat(messages, ANSWER_SCHEMA)
        generation.update(input=messages, output=actual['parsed'], usage_details={'input':actual['input_tokens'],'output':actual['output_tokens']})
    root.update(output=actual['parsed'])
    mini_trace_id = root.trace_id
lf.flush()
display(Markdown(f"[실제 생성 요청 열기]({lf.get_trace_url(trace_id=mini_trace_id)})"))

**확인 결과 / 화면에서 볼 것**

로그인한 브라우저에서 링크를 엽니다. root 아래 generation 하나가 보이고 실제 입력·출력·모델·토큰을 확인합니다. 이 최소 예제에는 검색·평가 단계가 아직 없습니다.

## 2. 검색·생성·검사·평가를 한 요청으로 연결

Q03a는 교환 기간과 비용을 함께 묻습니다. 실행 전에 기간·비용·조건이 모두 나올지 예상해 보세요.

In [ ]:
first = traced_question(lf, ollama, by_id['Q03a'])
print(first['answer'])
display(Markdown(f"[Q03a 전체 요청 열기]({first['url']})"))

**확인 결과 / 화면에서 볼 것**

전체 요청 아래 01-retrieval → 02-ollama-answer → 03-code-checks → 04-ollama-judge가 생깁니다. 원본 구현은 [langfuse_lab.py](../langfuse_lab.py)의 traced_question이며 각 실제 작업을 with 문으로 감쌉니다.

## 3. 서버 저장과 부모·자식 관계 확인

SDK 전송 성공과 서버 저장 완료를 구분합니다. 백그라운드 worker 처리 후 API로 다시 읽습니다.

In [ ]:
stored = wait_for_trace(lf, first)
display(pd.DataFrame(observation_table(stored)))
roots = [o for o in stored['observations'] if o['name'].startswith('rag-')]
assert len(roots) == 1
root_id = roots[0]['id']
assert sum(o.get('parentObservationId') == root_id for o in stored['observations']) == 4
print('실제 부모 1개 + 하위 작업 4개 확인')

**확인 결과 / 화면에서 볼 것**

요청 하나에 observations 5개. generation 2개에는 실제 토큰이 있습니다. 검색 문서는 retrieval Output, 모델에 보낸 메시지는 answer Input에서 확인합니다.

## 4. 평가 점수와 판단 근거 확인

코드 점수와 실제 모델 판정은 서로 다른 근거입니다. 모델이 supported라고 해도 코드 실패를 덮지 않습니다.

In [ ]:
display(pd.DataFrame(score_table(stored)))
print('참고 기준:', by_id['Q03a']['reference'])
print('실제 답변:', first['answer']['answer'])
print('실제 평가자:', first['judge'])

**확인 결과 / 화면에서 볼 것**

Trace의 Scores에서 BOOLEAN 검사와 CATEGORICAL llm_groundedness를 확인합니다. 숫자 1은 해당 코드 조건 통과이며 답변 정확도 100%가 아닙니다. 기간·비용·미착용·재고 조건을 직접 읽어봅니다.

## 5. 문서 밖 재고 질문의 두 응답 비교

“현재 재고가 없다”와 “재고를 확인할 수 없다”는 의미가 다릅니다. 어느 단계에서 추측했는지 찾습니다.

In [ ]:
baseline = traced_question(lf, ollama, by_id['Q24a'], prompt_version='baseline')
grounded = traced_question(lf, ollama, by_id['Q24a'], prompt_version='grounded')
for result in [baseline, grounded]:
    wait_for_trace(lf, result)
    print(result['prompt_version'], result['answer'], result['judge'])
    display(Markdown(f"[{result['prompt_version']} 요청 열기]({result['url']})"))

**확인 결과 / 화면에서 볼 것**

두 요청의 검색 문서와 모델이 실제로 보았던 Input을 비교합니다. 확인 불가 문장인데 abstain=false일 수 있습니다. 실제 판정은 실행마다 달라집니다.

## 6. 사람 검토의 기록

먼저 실제 답변을 읽고 아래 REVIEW를 직접 선택하세요. 기본값 None은 사람 평가를 수행하지 않았다는 뜻입니다.

In [ ]:
REVIEW = None  # 읽은 뒤 'supported', 'unsupported', 'needs_review' 중 선택
COMMENT = ''  # 판단 근거를 한 문장으로 작성
if REVIEW is None:
    print('사람 검토 미실행. 답변·근거를 읽은 뒤 REVIEW와 COMMENT를 채우세요.')
else:
    assert REVIEW in {'supported','unsupported','needs_review'} and COMMENT.strip()
    lf.create_score(trace_id=grounded['trace_id'], name='human_review', value=REVIEW,
                    data_type='CATEGORICAL', comment=COMMENT, score_id=grounded['trace_id']+'-human-review')
    lf.flush()
    print('사람 검토 전송. Scores를 새로고침하세요.')

**확인 결과 / 화면에서 볼 것**

미실행 평가를 사람 평가로 채우지 않습니다. 실제 검토 후 이름이 다른 human_review와 llm_groundedness를 비교합니다.

## 7. 실패 요청을 다음 실습으로 전달

가짜 재고 안내가 반복되지 않도록, 관찰한 실패를 회귀 평가 사례로 남깁니다.

In [ ]:
out = ROOT/'week11/lab/reports/langfuse'
out.mkdir(parents=True, exist_ok=True)
(out/'lesson09-traces.json').write_text(json.dumps([first,baseline,grounded], ensure_ascii=False, indent=2), encoding='utf-8')
print(out/'lesson09-traces.json')
print('다음 실습: 같은 질문 집합으로 세 버전을 비교한 뒤 배포 기준 적용')

**확인 결과 / 화면에서 볼 것**

파일은 본인 로컬 Git 제외 폴더에 저장됩니다. 10번은 독립 실행되므로 이 파일이 없어도 됩니다.

## 8. 관찰에서 개선으로 이어지는 질문

실험 전에 결론을 예측하되, 결과는 실제 기록으로 설명합니다.

In [ ]:
display(pd.DataFrame([
 {'관찰':'재고를 단정한 답변','검토':'검색 문서에 실시간 재고가 있는가','다음 실험':'기능 제한 정책'},
 {'관찰':'확인 불가 문장 + abstain=false','검토':'응답 계약이 의미와 일치하는가','다음 실험':'코드 검사 + 정책 분기'},
 {'관찰':'평가자도 supported','검토':'사람 기준과 독립적으로 대조했는가','다음 실험':'실패 사례 회귀 평가'},
]))
lf.flush()

**확인 결과 / 화면에서 볼 것**

모델에게 더 강하게 부탁하는 변경과, 애초에 제공하지 않는 기능을 차단하는 서비스 변경을 구분합니다.

## 마무리

연결이 끝나면 `lf.flush()`로 전송합니다. 실제 저장은 API 재조회로 확인합니다. 모델·프롬프트·기기 상태에 따라 답변과 지연은 달라집니다.
수업 결과는 합성 사례이며 일반적인 서비스 정확도 또는 운영 배포 승인이 아닙니다.

Docker 중지(데이터 유지): `docker compose -f week11/lab/langfuse/compose.yaml stop`
재개: 같은 경로의 `up -d --wait`. **`down -v`는 데이터를 삭제하므로 일반 종료에 사용하지 않습니다.**
[화면별 시연 안내](https://smilesjcha.github.io/llmops-16week/week11/lab/demo/) / [100분 진행·문제 해결](../langfuse/README.md) / [전체 실행 순서](../WALKTHROUGH.md)
